# 03. Exploratory Data Analysis

This notebook answers the first business questions defined in `docs/business_questions.md`.

Scope: Q1 overall sales performance, Q2 sales evolution over time, and Q4 product performance.
The analysis uses `orders_analytical.csv`, the analytical dataset generated by this project.

## Analytical conventions

- The current dataset has one row per cleaned order.
- Order counts use distinct `OrderID`.
- Monetary metrics exclude rows with missing financial inputs.
- These results describe associations and distributions; they do not establish causality.

## Analysis workflow

Each section follows the same sequence:

1. Define the business question.
2. Select the appropriate analytical grain.
3. Apply the documented missing-value rules.
4. Calculate the metrics with Pandas.
5. Visualize the main patterns.
6. Record observations separately from recommendations.

The first questions in this notebook are exploratory results. They will later be reproduced with SQL and translated into Power BI measures.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

project_root = Path.cwd()
while project_root != project_root.parent:
    if (project_root / "data" / "processed").exists():
        break
    project_root = project_root.parent

analytical_path = project_root / "data" / "processed" / "orders_analytical.csv"
orders = pd.read_csv(analytical_path, parse_dates=["OrderDate", "SignupDate"])
orders.head()

In [ ]:
expected_columns = {
    "OrderID", "CustomerID", "OrderDate", "Quantity",
    "Discount", "Status", "Category", "Sales", "OrderValue",
}
missing_columns = expected_columns.difference(orders.columns)
assert not missing_columns, f"Missing columns: {sorted(missing_columns)}"
assert orders["OrderID"].is_unique
print(f"Rows: {len(orders):,}")
print(f"Distinct orders: {orders['OrderID'].nunique():,}")
print(f"Missing Sales: {orders['Sales'].isna().sum():,}")
print(f"Missing OrderValue: {orders['OrderValue'].isna().sum():,}")

## Q1. Overall sales performance

The monetary metrics below use only rows where the required financial values are available.

In [ ]:
financial_orders = orders.dropna(subset=["Sales", "OrderValue"]).copy()
sales_kpis = pd.Series({
    "distinct_orders": financial_orders["OrderID"].nunique(),
    "units_sold": financial_orders["Quantity"].sum(),
    "sales_value": financial_orders["Sales"].sum(),
    "order_value": financial_orders["OrderValue"].sum(),
    "aov": financial_orders.groupby("OrderID")["OrderValue"].sum().mean(),
})
sales_kpis

## Q2. Sales evolution over time

Monthly metrics are calculated after excluding records without a valid order date or monetary value.

In [ ]:
monthly_orders = financial_orders.dropna(subset=["OrderDate"]).copy()
monthly_orders["OrderMonth"] = monthly_orders["OrderDate"].dt.to_period("M").dt.to_timestamp()
monthly_sales = (
    monthly_orders.groupby("OrderMonth")
    .agg(
        distinct_orders=("OrderID", "nunique"),
        units_sold=("Quantity", "sum"),
        sales_value=("Sales", "sum"),
        order_value=("OrderValue", "sum"),
    )
)
monthly_sales["aov"] = monthly_sales["order_value"] / monthly_sales["distinct_orders"]
monthly_sales.head()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
monthly_sales["sales_value"].plot(ax=axes[0], title="Monthly Sales Value")
monthly_sales["distinct_orders"].plot(ax=axes[1], title="Monthly Distinct Orders")
axes[0].set_ylabel("Sales")
axes[1].set_ylabel("Orders")
plt.tight_layout()

## Q4. Product and category performance

This ranking compares categories by sales value and unit volume.

In [ ]:
category_performance = (
    financial_orders.groupby("Category")
    .agg(
        sales_value=("Sales", "sum"),
        order_value=("OrderValue", "sum"),
        units_sold=("Quantity", "sum"),
        distinct_orders=("OrderID", "nunique"),
    )
    .sort_values("order_value", ascending=False)
)
category_performance["sales_share"] = category_performance["order_value"] / category_performance["order_value"].sum()
category_performance

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
category_performance["order_value"].plot(kind="bar", ax=ax)
ax.set_title("Order Value by Category")
ax.set_ylabel("Order Value")
ax.set_xlabel("Category")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()

## Q5. Product volume versus sales value

This analysis compares product demand with financial contribution. Monetary metrics exclude rows with missing financial values, while unit metrics exclude rows with missing quantity.

In [ ]:
product_performance = (
    financial_orders.groupby(["ProductID", "ProductName", "Category"])
    .agg(
        sales_value=("Sales", "sum"),
        order_value=("OrderValue", "sum"),
        units_sold=("Quantity", "sum"),
        distinct_orders=("OrderID", "nunique"),
        average_unit_price=("UnitPrice", "mean"),
    )
    .sort_values("order_value", ascending=False)
)

product_performance["sales_share"] = (
    product_performance["order_value"]
    / product_performance["order_value"].sum()
)
product_performance.head(10)

In [ ]:
unit_performance = (
    orders.dropna(subset=["Quantity"])
    .groupby(["ProductID", "ProductName", "Category"])
    .agg(
        units_sold=("Quantity", "sum"),
        distinct_orders=("OrderID", "nunique"),
        sales_value=("Sales", "sum"),
    )
    .sort_values("units_sold", ascending=False)
)

unit_performance["volume_rank"] = unit_performance["units_sold"].rank(
    method="dense", ascending=False
)
product_performance["sales_rank"] = product_performance["order_value"].rank(
    method="dense", ascending=False
)

rank_comparison = product_performance[["sales_rank"]].join(
    unit_performance[["volume_rank"]],
    how="inner",
).sort_values("sales_rank")
rank_comparison.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
product_performance.head(10).sort_values("order_value").plot(
    y="order_value",
    kind="barh",
    ax=axes[0],
    legend=False,
)
unit_performance.head(10).sort_values("units_sold").plot(
    y="units_sold",
    kind="barh",
    ax=axes[1],
    legend=False,
)
axes[0].set_title("Top Products by Order Value")
axes[1].set_title("Top Products by Units Sold")
axes[0].set_xlabel("Order Value")
axes[1].set_xlabel("Units Sold")
plt.tight_layout()

## Initial observations

Complete this section after reviewing the generated tables and charts. Separate observed patterns from recommendations, and document any exclusions caused by missing values.

## Q7. Customer purchasing behavior and concentration

Customer-level metrics are calculated after aggregating orders by `CustomerID`. Orders without a matching customer record are excluded from customer-attribute analysis and reported separately.

In [ ]:
customer_orders = orders.dropna(subset=["CustomerID"]).copy()
customer_summary = (
    customer_orders.groupby("CustomerID")
    .agg(
        distinct_orders=("OrderID", "nunique"),
        units_sold=("Quantity", "sum"),
        sales_value=("Sales", "sum"),
        order_value=("OrderValue", "sum"),
    )
)
customer_summary["customer_type"] = customer_summary["distinct_orders"].map(
    lambda order_count: "Repeat" if order_count > 1 else "One-time"
)
customer_summary = customer_summary.sort_values("order_value", ascending=False)

customer_kpis = pd.Series({
    "purchasing_customers": customer_summary.index.nunique(),
    "repeat_customers": (customer_summary["distinct_orders"] > 1).sum(),
    "repeat_customer_rate": (customer_summary["distinct_orders"] > 1).mean(),
    "top_10_sales_share": (
        customer_summary["order_value"].head(10).sum()
        / customer_summary["order_value"].sum()
    ),
})
customer_kpis

In [ ]:
customer_orders["CustomerID"].value_counts().head(10).sort_values().plot(
    kind="barh",
    figsize=(10, 5),
    title="Top Customers by Number of Orders",
)
plt.xlabel("Distinct Orders")
plt.tight_layout()

## Q8. One-time versus repeat customers

This comparison uses distinct order counts per customer. Monetary metrics are based on available `OrderValue` values.

In [ ]:
customer_type_performance = (
    customer_summary.groupby("customer_type")
    .agg(
        customers=("distinct_orders", "size"),
        total_orders=("distinct_orders", "sum"),
        sales_value=("sales_value", "sum"),
        order_value=("order_value", "sum"),
    )
)
customer_type_performance["aov"] = (
    customer_type_performance["order_value"]
    / customer_type_performance["total_orders"]
)
customer_type_performance["sales_share"] = (
    customer_type_performance["order_value"]
    / customer_type_performance["order_value"].sum()
)
customer_type_performance

## Q9. Customer segments and cities

Segment and city comparisons use customer attributes from the analytical dataset. Missing attributes remain visible as `Unknown` so their impact is not silently removed.

In [ ]:
segment_orders = orders.copy()
segment_orders["CustomerSegment"] = segment_orders["CustomerSegment"].fillna("Unknown")
segment_orders["City"] = segment_orders["City"].fillna("Unknown")

segment_performance = (
    segment_orders.dropna(subset=["OrderValue"])
    .groupby("CustomerSegment")
    .agg(
        customers=("CustomerID", "nunique"),
        distinct_orders=("OrderID", "nunique"),
        order_value=("OrderValue", "sum"),
    )
    .sort_values("order_value", ascending=False)
)
segment_performance["aov"] = (
    segment_performance["order_value"]
    / segment_performance["distinct_orders"]
)
segment_performance["sales_share"] = (
    segment_performance["order_value"]
    / segment_performance["order_value"].sum()
)
segment_performance

In [ ]:
city_performance = (
    segment_orders.dropna(subset=["OrderValue"])
    .groupby("City")
    .agg(
        customers=("CustomerID", "nunique"),
        distinct_orders=("OrderID", "nunique"),
        order_value=("OrderValue", "sum"),
    )
    .sort_values("order_value", ascending=False)
)
city_performance["sales_share"] = (
    city_performance["order_value"]
    / city_performance["order_value"].sum()
)
city_performance.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
segment_performance["order_value"].sort_values().plot(
    kind="barh",
    ax=axes[0],
    title="Order Value by Customer Segment",
)
city_performance.head(10)["order_value"].sort_values().plot(
    kind="barh",
    ax=axes[1],
    title="Top Cities by Order Value",
)
axes[0].set_xlabel("Order Value")
axes[1].set_xlabel("Order Value")
plt.tight_layout()

## Customer analysis observations

Use the tables and charts above to record observations before recommendations. Include the repeat-customer share, customer concentration, segment differences, city rankings, and the treatment of unmatched or missing customer attributes.

## Q12. Payment and order reconciliation

This section validates whether payment records can be safely connected to orders. It checks key uniqueness, unmatched identifiers, payment records per order, and payment-status distribution.

In [ ]:
orders_for_reconciliation = pd.read_csv(
    project_root / "data" / "processed" / "orders_processed.csv"
)
payments_for_reconciliation = pd.read_csv(
    project_root / "data" / "processed" / "payments_processed.csv"
)

assert orders_for_reconciliation["OrderID"].is_unique
assert payments_for_reconciliation["PaymentID"].is_unique

payment_order_reconciliation = payments_for_reconciliation.merge(
    orders_for_reconciliation[["OrderID"]],
    on="OrderID",
    how="outer",
    indicator=True,
)

reconciliation_summary = pd.Series({
    "orders": orders_for_reconciliation["OrderID"].nunique(),
    "payments": payments_for_reconciliation["PaymentID"].nunique(),
    "orders_with_payment": payments_for_reconciliation["OrderID"].nunique(),
    "payments_per_order": (
        payments_for_reconciliation.groupby("OrderID").size().mean()
    ),
    "unmatched_payment_records": (
        payment_order_reconciliation["_merge"].eq("left_only").sum()
    ),
    "orders_without_payment": (
        payment_order_reconciliation["_merge"].eq("right_only").sum()
    ),
})
reconciliation_summary

In [ ]:
payment_status_summary = (
    payments_for_reconciliation["PaymentStatus"]
    .value_counts(dropna=False)
    .rename_axis("payment_status")
    .reset_index(name="payment_records")
)
payment_status_summary["share"] = (
    payment_status_summary["payment_records"]
    / payment_status_summary["payment_records"].sum()
)
payment_status_summary

In [ ]:
payment_status_summary.plot(
    x="payment_status",
    y="payment_records",
    kind="bar",
    legend=False,
    figsize=(8, 5),
    title="Payment Status Distribution",
)
plt.ylabel("Payment Records")
plt.xlabel("Payment Status")
plt.xticks(rotation=0)
plt.tight_layout()

### Reconciliation interpretation

A matched order and payment record confirms referential consistency only. Payment status must be analyzed separately before drawing conclusions about successful transactions, failures, or refunds.